# Feature engineering and ML-ready dataset
Input: `ChargingRecords_cleaned.csv` (from the Phase 1 cleaning notebook). Run the cells top to bottom.

**Prediction task assumed:** predict the `Demand` of a charging session *at the moment it starts*. Everything the model sees must therefore be known at the start time.

Steps:
1. Load data and build exact timestamps
2. Time features: hour, day, month, day of week, weekend, time of day
3. Historical station (charger) utilization features, built only from sessions that had already ended
4. Verify the historical features by brute force
5. Decide which columns are leakage and must be excluded
6. Chronological train / validation / test split
7. Encode categorical features (fitted on the training set only)
8. Save the final datasets
9. Sanity check with a quick baseline model

## 1. Load data and build exact timestamps
The minute-level `StartDatetime` loses the seconds, so rebuild second-level timestamps from the Day and Time columns.

In [1]:
import json
import numpy as np
import pandas as pd

df = pd.read_csv(r"D:\HCL_PROJECTS\EV_Charging_Demand_Analysis\ChargingRecords_preprocessed.csv")

df["StartTS"] = pd.to_datetime(df["StartDay"] + " " + df["StartTime"], format="%d-%m-%Y %H:%M:%S")
df["EndTS"]   = pd.to_datetime(df["EndDay"]   + " " + df["EndTime"],   format="%d-%m-%Y %H:%M:%S")

df = df.sort_values(["StartTS", "ChargerID"]).reset_index(drop=True)
print(df.shape)
print(df["StartTS"].min(), "->", df["StartTS"].max())
df[["ChargerID", "StartTS", "EndTS", "Duration", "Demand"]].head()

(72031, 32)
2021-09-30 17:14:26 -> 2022-09-30 23:11:59


,ChargerID,StartTS,EndTS,Duration,Demand
0,291,2021-09-30 17:14:26,2021-10-01 01:21:28,487,27.68
1,191,2021-09-30 18:22:36,2021-10-01 02:21:03,478,55.13
2,320,2021-09-30 18:57:46,2021-10-01 00:44:59,347,38.45
3,433,2021-09-30 19:52:22,2021-10-01 00:21:51,269,30.17
4,250,2021-09-30 20:48:46,2021-10-01 03:35:11,406,47.00


## 2. Time features
Taken from the session **start** time. Cyclical encodings (sin / cos) let a model see that hour 23 is next to hour 0, Sunday is next to Monday, and December is next to January.

Time of day bands: Night 00-05, Morning 06-11, Afternoon 12-17, Evening 18-23.

In [2]:
ts = df["StartTS"]

df["Hour"]       = ts.dt.hour
df["Day"]        = ts.dt.day                 # day of month
df["Month"]      = ts.dt.month
df["DayOfWeek"]  = ts.dt.dayofweek           # Monday = 0
df["IsWeekend"]  = (df["DayOfWeek"] >= 5).astype(int)

df["TimeOfDay"] = pd.cut(df["Hour"], bins=[-1, 5, 11, 17, 23],
                         labels=["Night", "Morning", "Afternoon", "Evening"]).astype(str)

df["Hour_sin"]  = np.sin(2 * np.pi * df["Hour"] / 24)
df["Hour_cos"]  = np.cos(2 * np.pi * df["Hour"] / 24)
df["DoW_sin"]   = np.sin(2 * np.pi * df["DayOfWeek"] / 7)
df["DoW_cos"]   = np.cos(2 * np.pi * df["DayOfWeek"] / 7)
df["Month_sin"] = np.sin(2 * np.pi * (df["Month"] - 1) / 12)
df["Month_cos"] = np.cos(2 * np.pi * (df["Month"] - 1) / 12)

df[["StartTS", "Hour", "Day", "Month", "DayOfWeek", "IsWeekend", "TimeOfDay"]].head()

,StartTS,Hour,Day,Month,DayOfWeek,IsWeekend,TimeOfDay
0,2021-09-30 17:14:26,17,30,9,3,0,Afternoon
1,2021-09-30 18:22:36,18,30,9,3,0,Evening
2,2021-09-30 18:57:46,18,30,9,3,0,Evening
3,2021-09-30 19:52:22,19,30,9,3,0,Evening
4,2021-09-30 20:48:46,20,30,9,3,0,Evening


In [3]:
df["TimeOfDay"].value_counts()

TimeOfDay
Afternoon    26932
Evening      24663
Morning      18022
Night         2414
Name: count, dtype: int64

## 3. Historical station (charger) utilization features
For every session, these features describe the **same charger's past**, using only sessions that had **already ended** before this session started. A session that was still running at the start time is excluded, because its demand and end time are not known yet.

| Feature | Meaning |
|---|---|
| `hist_n_prior` | Number of earlier sessions on this charger |
| `hist_mean_demand_prior` | Mean demand of those earlier sessions |
| `hist_n_7d` | Sessions that ended in the last 7 days |
| `hist_mean_demand_7d` | Mean demand of those sessions |
| `hist_util_24h` | Share of the last 24 hours the charger was plugged in (plug-in minutes of sessions that ended in the last 24 h, divided by 1,440, capped at 1) |
| `hist_hours_since_last` | Hours since the charger's last session ended (capped at 720) |
| `has_history` | 1 if the charger has any earlier session |

Method: build a running total per charger over its session end times, then look up the running total at each session's start time (and at start minus 7 days or 24 hours) with `merge_asof`. The difference gives the window totals without any slow row-by-row loop. `allow_exact_matches=False` means a session ending at exactly the start time is not counted.

In [4]:
# Running totals per charger, ordered by session END time
ev = (df[["ChargerID", "EndTS", "Demand", "ElapsedMin"]]
        .sort_values("EndTS")
        .reset_index(drop=True))
g = ev.groupby("ChargerID")
ev["cum_n"]      = g.cumcount() + 1
ev["cum_demand"] = g["Demand"].cumsum()
ev["cum_busy"]   = g["ElapsedMin"].cumsum()
ev = ev.rename(columns={"EndTS": "EndKey"})

def totals_at(times):
    """Running totals of sessions that ended strictly before `times`, for each row (same charger)."""
    left = pd.DataFrame({"ChargerID": df["ChargerID"], "t": times}).reset_index().sort_values("t")
    out = pd.merge_asof(left, ev, left_on="t", right_on="EndKey", by="ChargerID",
                        allow_exact_matches=False)
    out = out.set_index("index").sort_index()
    for c in ["cum_n", "cum_demand", "cum_busy"]:
        out[c] = out[c].fillna(0)
    return out

now  = totals_at(df["StartTS"])
d7   = totals_at(df["StartTS"] - pd.Timedelta(days=7))
d1   = totals_at(df["StartTS"] - pd.Timedelta(hours=24))

n_win = now["cum_n"] - d7["cum_n"]
df["hist_n_prior"]            = now["cum_n"]
df["hist_mean_demand_prior"]  = now["cum_demand"] / now["cum_n"].where(now["cum_n"] > 0)
df["hist_n_7d"]               = n_win
df["hist_mean_demand_7d"]     = (now["cum_demand"] - d7["cum_demand"]) / n_win.where(n_win > 0)
df["hist_util_24h"]           = ((now["cum_busy"] - d1["cum_busy"]) / 1440).clip(0, 1)
df["hist_hours_since_last"]   = ((df["StartTS"] - now["EndKey"]).dt.total_seconds() / 3600).clip(upper=720)
df["has_history"]             = (now["cum_n"] > 0).astype(int)

hist_cols = ["hist_n_prior", "hist_mean_demand_prior", "hist_n_7d", "hist_mean_demand_7d",
             "hist_util_24h", "hist_hours_since_last", "has_history"]
print("Share of sessions with no earlier session on the charger:", round(1 - df["has_history"].mean(), 3))
df[hist_cols].describe().round(2)

Share of sessions with no earlier session on the charger: 0.029


,hist_n_prior,hist_mean_demand_prior,hist_n_7d,hist_mean_demand_7d,hist_util_24h,hist_hours_since_last,has_history
count,72031.00,69915.00,72031.00,60036.00,72031.00,69915.00,72031.00
mean,98.53,17.29,4.78,17.86,0.09,91.80,0.97
std,117.12,6.98,4.43,9.35,0.14,164.74,0.17
min,0.00,0.01,0.00,0.01,0.00,0.00,0.00
25%,14.00,12.93,1.00,11.49,0.00,10.16,1.00
50%,52.00,16.21,4.00,16.33,0.00,23.83,1.00
75%,140.00,21.17,7.00,22.45,0.14,81.33,1.00
max,669.00,95.34,26.00,95.34,1.00,720.00,1.00


In [5]:
# Missing values are expected for chargers with no history yet; they are filled later, using training data only
df[hist_cols].isna().sum()

hist_n_prior                  0
hist_mean_demand_prior     2116
hist_n_7d                     0
hist_mean_demand_7d       11995
hist_util_24h                 0
hist_hours_since_last      2116
has_history                   0
dtype: int64

## 4. Verify the historical features by brute force
Recompute the features for random sessions with a plain loop over the raw data, and check they match.

In [6]:
rng = np.random.default_rng(0)
sample_idx = rng.choice(len(df), 300, replace=False)
by_charger = {k: v for k, v in df.groupby("ChargerID")}

bad = 0
for i in sample_idx:
    r = df.loc[i]
    c = by_charger[r["ChargerID"]]
    past = c[c["EndTS"] < r["StartTS"]]                                  # already ended
    w7   = past[past["EndTS"] >= r["StartTS"] - pd.Timedelta(days=7)]
    w1   = past[past["EndTS"] >= r["StartTS"] - pd.Timedelta(hours=24)]
    exp = {
        "hist_n_prior": len(past),
        "hist_mean_demand_prior": past["Demand"].mean() if len(past) else np.nan,
        "hist_n_7d": len(w7),
        "hist_mean_demand_7d": w7["Demand"].mean() if len(w7) else np.nan,
        "hist_util_24h": min(w1["ElapsedMin"].sum() / 1440, 1.0),
        "hist_hours_since_last": min((r["StartTS"] - past["EndTS"].max()).total_seconds() / 3600, 720) if len(past) else np.nan,
    }
    for k, v in exp.items():
        got = r[k]
        if not ((pd.isna(v) and pd.isna(got)) or np.isclose(v, got, atol=1e-6)):
            bad += 1
            print("Mismatch", i, k, v, got)

print("Rows checked:", len(sample_idx), "| mismatches:", bad)
assert bad == 0

Rows checked: 300 | mismatches: 0


## 5. Columns that must not be used as features (leakage)
At the start of a session the model cannot know how it will end. These columns are therefore excluded:

- `Duration`, `ElapsedMin`, `IdleMin`, `FlagIdleOver30`: all describe how long the session lasted (`Duration` alone has a 0.66 correlation with `Demand`).
- `EndDay`, `EndTime`, `EndDatetime`, `EndTS`: the end time.
- The raw start columns (`StartDay`, `StartTime`, `StartDatetime`, `StartTS`) are replaced by the engineered time features.

High-cardinality identifiers are also left out:
- `ChargerID` has 2,115 values; the history features carry what is useful about each charger.
- `UserID` has 2,320 values, and ID 0 alone covers 43% of sessions (likely a placeholder), so it is not used as a feature.

In [7]:
target = "Demand"

numeric_features = [
    "Hour", "Day", "Month", "DayOfWeek", "IsWeekend",
    "Hour_sin", "Hour_cos", "DoW_sin", "DoW_cos", "Month_sin", "Month_cos",
    "ChargerType", "ChargerCompany",
] + hist_cols
categorical_features = ["Location", "TimeOfDay"]

print(len(numeric_features), "numeric features,", len(categorical_features), "categorical features")

20 numeric features, 2 categorical features


## 6. Leakage-safe split: chronological, with a purge gap
Sessions are ordered by start time: the first 70% train, the next 15% validation, the last 15% test. A random split would let the model learn from the future.

A session that started in one period but ended in the next is **dropped** from the earlier period (a purge), so no training session's outcome is only known after the validation period starts.

In [8]:
n = len(df)
t_val  = df["StartTS"].iloc[int(n * 0.70)]
t_test = df["StartTS"].iloc[int(n * 0.85)]

is_train = (df["StartTS"] < t_val)  & (df["EndTS"] <= t_val)
is_val   = (df["StartTS"] >= t_val) & (df["StartTS"] < t_test) & (df["EndTS"] <= t_test)
is_test  = df["StartTS"] >= t_test

train, val, test = df[is_train].copy(), df[is_val].copy(), df[is_test].copy()

print("Validation starts:", t_val)
print("Test starts:      ", t_test)
print(f"Train {len(train)} | Val {len(val)} | Test {len(test)} | Purged {n - len(train) - len(val) - len(test)}")

# Checks
assert train["EndTS"].max() <= val["StartTS"].min()
assert val["EndTS"].max()   <= test["StartTS"].min()

Validation starts: 2022-07-02 18:49:19
Test starts:       2022-08-13 15:07:07
Train 50391 | Val 10776 | Test 10805 | Purged 59


In [9]:
# Compare the three periods
pd.DataFrame({
    "sessions":    [len(train), len(val), len(test)],
    "mean_demand": [round(train[target].mean(), 2), round(val[target].mean(), 2), round(test[target].mean(), 2)],
    "first_start": [train["StartTS"].min(), val["StartTS"].min(), test["StartTS"].min()],
    "last_start":  [train["StartTS"].max(), val["StartTS"].max(), test["StartTS"].max()],
}, index=["train", "val", "test"])

,sessions,mean_demand,first_start,last_start
train,50391,17.44,2021-09-30 17:14:26,2022-07-02 18:11:49
val,10776,17.90,2022-07-02 18:49:19,2022-08-13 13:55:05
test,10805,17.78,2022-08-13 15:07:07,2022-09-30 23:11:59


In [10]:
# The data covers only one year, so validation and test months (Jul to Sep) are mostly absent from training
for name, part in [("train", train), ("val", val), ("test", test)]:
    print(name, sorted(int(m) for m in part["Month"].unique()))

train [1, 2, 3, 4, 5, 6, 7, 9, 10, 11, 12]
val [7, 8]
test [8, 9]


## 7. Fill missing values and encode categories (fitted on the training set only)
- Missing demand means (chargers with no history) are filled with the **training** mean demand.
- `hist_hours_since_last` with no earlier session is filled with its cap of 720 hours.
- `Location` and `TimeOfDay` are one-hot encoded. The category lists come from the training set; an unseen category in validation or test becomes all zeros instead of causing an error.
- Tree-based models need no scaling. For linear models or neural networks, fit a `StandardScaler` on the training set only and apply it to the other two sets.

In [11]:
from sklearn.preprocessing import OneHotEncoder

# Fill values come from the training set only
fill_values = {
    "hist_mean_demand_prior": train[target].mean(),
    "hist_mean_demand_7d":    train[target].mean(),
    "hist_hours_since_last":  720.0,
}

def fill(part):
    part = part.copy()
    for col, value in fill_values.items():
        part[col] = part[col].fillna(value)
    return part

train, val, test = fill(train), fill(val), fill(test)

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
encoder.fit(train[categorical_features])
encoded_names = list(encoder.get_feature_names_out(categorical_features))

def build(part):
    enc = pd.DataFrame(encoder.transform(part[categorical_features]),
                       columns=encoded_names, index=part.index)
    X = pd.concat([part[numeric_features], enc], axis=1)
    return X, part[target]

X_train, y_train = build(train)
X_val,   y_val   = build(val)
X_test,  y_test  = build(test)

print("Feature matrix:", X_train.shape, X_val.shape, X_test.shape)
print("Missing values:", int(X_train.isna().sum().sum() + X_val.isna().sum().sum() + X_test.isna().sum().sum()))
print("Columns identical across sets:", list(X_train.columns) == list(X_val.columns) == list(X_test.columns))
X_train.head()

Feature matrix: (50391, 38) (10776, 38) (10805, 38)
Missing values: 0
Columns identical across sets: True


,Hour,Day,Month,DayOfWeek,IsWeekend,Hour_sin,Hour_cos,DoW_sin,DoW_cos,Month_sin,...,Location_public area,Location_public institution,Location_public parking lot,Location_resort,Location_restaurant,Location_sightseeing,TimeOfDay_Afternoon,TimeOfDay_Evening,TimeOfDay_Morning,TimeOfDay_Night
0,17,30,9,3,0,-0.965926,-2.588190e-01,0.433884,-0.900969,-0.866025,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1,18,30,9,3,0,-1.000000,-1.836970e-16,0.433884,-0.900969,-0.866025,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
2,18,30,9,3,0,-1.000000,-1.836970e-16,0.433884,-0.900969,-0.866025,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,19,30,9,3,0,-0.965926,2.588190e-01,0.433884,-0.900969,-0.866025,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,20,30,9,3,0,-0.866025,5.000000e-01,0.433884,-0.900969,-0.866025,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


## 8. Save the final datasets

In [12]:
train_out = pd.concat([X_train, y_train], axis=1)
val_out   = pd.concat([X_val,   y_val],   axis=1)
test_out  = pd.concat([X_test,  y_test],  axis=1)

train_out.to_csv("train.csv", index=False)
val_out.to_csv("val.csv",   index=False)
test_out.to_csv("test.csv", index=False)

meta = {
    "target": target,
    "features": list(X_train.columns),
    "split": {
        "validation_starts": str(t_val),
        "test_starts": str(t_test),
        "rows": {"train": len(train_out), "val": len(val_out), "test": len(test_out)},
    },
    "fill_values_from_train": {k: float(v) for k, v in fill_values.items()},
    "one_hot_categories": {c: list(cats) for c, cats in zip(categorical_features, encoder.categories_)},
    "excluded_for_leakage": ["Duration", "ElapsedMin", "IdleMin", "FlagIdleOver30",
                              "EndDay", "EndTime", "EndDatetime", "EndTS"],
    "excluded_identifiers": ["UserID", "ChargerID"],
}
with open("feature_metadata.json", "w") as f:
    json.dump(meta, f, indent=2)

print("Saved train.csv, val.csv, test.csv and feature_metadata.json")

Saved train.csv, val.csv, test.csv and feature_metadata.json


## 9. Sanity check with a quick baseline
Train a small model on the training set and score it on the validation set. This is only a check that the features carry signal and that nothing breaks; it is not the final model.

In [13]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

baseline_pred = np.full(len(y_val), y_train.mean())
print(f"Predict-the-mean baseline : MAE {mean_absolute_error(y_val, baseline_pred):.2f} | R2 {r2_score(y_val, baseline_pred):.3f}")

model = HistGradientBoostingRegressor(random_state=0)
model.fit(X_train, y_train)
pred = model.predict(X_val)
print(f"Gradient boosting         : MAE {mean_absolute_error(y_val, pred):.2f} | R2 {r2_score(y_val, pred):.3f}")

Predict-the-mean baseline : MAE 10.84 | R2 -0.001
Gradient boosting         : MAE 8.92 | R2 0.262


In [14]:
# Which features matter most? (permutation importance on a validation sample)
from sklearn.inspection import permutation_importance

sample = X_val.sample(min(5000, len(X_val)), random_state=0)
imp = permutation_importance(model, sample, y_val.loc[sample.index], n_repeats=3, random_state=0, scoring="neg_mean_absolute_error")
(pd.Series(imp.importances_mean, index=X_val.columns).sort_values(ascending=False).head(12).round(3)
   .rename("MAE increase when shuffled").to_frame())

,MAE increase when shuffled
hist_mean_demand_prior,1.554
Hour_cos,0.589
ChargerType,0.218
hist_mean_demand_7d,0.132
ChargerCompany,0.079
hist_hours_since_last,0.062
Hour_sin,0.061
Hour,0.060
hist_n_prior,0.056
hist_util_24h,0.036


## 10. Do the historical features help? A quick comparison
The same model is trained with and without the historical features, and scored on both the validation and the test sets. The month features are also dropped in the second and third rows: the year of data means validation and test months are unseen in training, and they add nothing to the score, so you may prefer to leave them out of the final model.

In [15]:
month_cols = ["Month", "Month_sin", "Month_cos"]
hist_feature_cols = hist_cols

def score(drop, label):
    cols = [c for c in X_train.columns if c not in drop]
    m = HistGradientBoostingRegressor(random_state=0).fit(X_train[cols], y_train)
    row = {"setup": label, "n_features": len(cols)}
    for name, X, y in [("val", X_val, y_val), ("test", X_test, y_test)]:
        p = m.predict(X[cols])
        row[f"{name}_MAE"] = round(mean_absolute_error(y, p), 2)
        row[f"{name}_R2"]  = round(r2_score(y, p), 3)
    return row

pd.DataFrame([
    score([],                                 "all features"),
    score(month_cols,                         "without month features"),
    score(month_cols + hist_feature_cols,     "without month and history features"),
]).set_index("setup")

,n_features,val_MAE,val_R2,test_MAE,test_R2
setup,,,,,
all features,38,8.92,0.262,8.96,0.233
without month features,35,8.93,0.263,8.96,0.228
without month and history features,28,9.86,0.139,9.92,0.098


### The result
- The historical charger features are the most useful addition: removing them raises the validation error by about 11% and halves R2.
- Month features change nothing, as expected with only one year of data.
- R2 is modest (about 0.25) because the biggest driver of demand, how long the session lasts, is not known when a session starts and is excluded on purpose. Including `Duration` would score much higher, but only because it would leak the outcome.